# 11g — Dipole Maps, Fraction vs Time, and Rose Plots per DDF

## Purpose

This notebook generates three families of diagnostic figures from the DIA alert data
previously downloaded by notebook `01_fink_block_flatlightcurves.ipynb` and stored
under `data_FINK_BLOCK_LC_01/`.

**No API calls are made here** — all data are read from local parquet / CSV files.

### Figures produced

#### (a) Alert count maps and dipole sky maps per DDF
For each Deep Drilling Field:
- Distribution of all alerts in (RA, Dec) — coloured by Gaia classification group.
- Overlay of dipole alerts as arrows (direction = `dipoleAngle`, length ∝ `dipoleLength`),
  coloured by filter band.

#### (b) Dipole fraction vs MJD (and calendar date YYYY-MM-DD)
- Per-night (1-day bin) dipole fraction for all DDFs stacked in one multi-panel figure.
- Twin axis: N alerts per bin (grey bars, log scale).
- Calendar-date axis on top.

#### (c) Rose plots per DDF
- Polar rose diagram of `dipoleAngle` distribution (0–360°, N through E).
- All filter bands overlaid with distinct colours inside the **same rose plot**
  (stacked angular histogram).
- Dashed uniform-distribution reference circle.

---
- **Author:** Sylvie Dagoret-Campagne — IJCLab / IN2P3 / CNRS — Université Paris-Saclay
- **Creation date:** 2026-05-24
- **Data source:** `data_FINK_BLOCK_LC_01/` (notebook 01 outputs)
- **Reference notebooks:** `11e_dipoleAngle.ipynb`, `08_Dipoles/01c_fink_dipoles_per_ddf.ipynb`


## 1. Imports & configuration

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
from astropy.time import Time

warnings.filterwarnings("ignore")
print(f"pandas {pd.__version__}  |  numpy {np.__version__}")

In [ ]:
# Enable interactive matplotlib backend
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found → interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found → falling back to %matplotlib inline")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# PATHS — data produced by notebook 01
# ─────────────────────────────────────────────────────────────────────────────
NB_TAG = "11g"
DIR_DATA_IN = "data_FINK_BLOCK_LC_01"  # read-only: outputs of notebook 01
DIR_FIGS = f"figs_FINK_BLOCK_LC_{NB_TAG}"
os.makedirs(DIR_FIGS, exist_ok=True)

# Key input files
FILE_DIAOBJ = os.path.join(DIR_DATA_IN, "diaobj_catalogue.csv")
FILE_VISIT = os.path.join(DIR_DATA_IN, "visit_index.csv")

# Per-category source parquets (detections = diaSources)
SRC_FILES = {
    "gaia_star_stable_hq": os.path.join(DIR_DATA_IN, "gaia_star_stable_hq_src.parquet"),
    "gaia_nophotgstar_stable_unknown_parallax": os.path.join(
        DIR_DATA_IN, "gaia_nophotgstar_stable_unknown_parallax_src.parquet"
    ),
    "gaia_star_variable": os.path.join(DIR_DATA_IN, "gaia_star_variable_src.parquet"),
}

# ─────────────────────────────────────────────────────────────────────────────
# PARAMETERS
# ─────────────────────────────────────────────────────────────────────────────
MJD_BIN_DAYS = 1  # time-bin width for the fraction-vs-time figures
N_BINS_ROSE = 36  # angular bins for rose plots (10° per bin)
MAX_ARROWS = 3000  # cap on arrows drawn per DDF sky map (performance)
SAVE_FIGS = True

BAND_ORDER = list("ugrizy")
BAND_COLORS = {
    "u": "#9b59b6",
    "g": "#2ecc71",
    "r": "#e74c3c",
    "i": "#e67e22",
    "z": "#3498db",
    "y": "#795548",
}

# Gaia group colours (for alert-count scatter)
GROUP_COLORS = {
    "gaia_star_stable_hq": "#1f77b4",
    "gaia_nophotgstar_stable_unknown_parallax": "#ff7f0e",
    "gaia_star_variable": "#d62728",
}
GROUP_LABELS = {
    "gaia_star_stable_hq": "Gaia stable HQ",
    "gaia_nophotgstar_stable_unknown_parallax": "Gaia stable (no G-mag)",
    "gaia_star_variable": "Gaia variable",
}

# LSST Deep Drilling Fields (RA/Dec J2000) — same as notebook 01
DEEP_FIELDS = {
    "COSMOS": (150.1191, 2.2058),
    "ELAIS-S1": (9.4500, -44.000),
    "XMM-LSS": (35.7080, -4.750),
    "ECDFS": (53.1250, -27.800),
    "EDFS-a": (58.9000, -49.315),
    "EDFS-b": (63.6000, -47.600),
    "EDFS": (61.2400, -48.423),
    "M49": (187.4000, 8.000),
}

plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "font.size": 9,
    }
)


def savefig(name: str):
    """Save current figure to DIR_FIGS in both PDF and PNG."""
    if SAVE_FIGS:
        for ext in ("pdf", "png"):
            plt.savefig(os.path.join(DIR_FIGS, f"{name}.{ext}"), bbox_inches="tight")
        print(f"  -> saved {name}.{{pdf,png}}")


print(f"Data  : {os.path.abspath(DIR_DATA_IN)}")
print(f"Figs  : {os.path.abspath(DIR_FIGS)}")

## 2. Utility functions

In [ ]:
def mjd_to_datestr(mjd_array) -> list:
    """Convert an array of MJD (TAI) values to ISO date strings 'YYYY-MM-DD'."""
    t = Time(np.asarray(mjd_array, dtype=float), format="mjd", scale="tai")
    return [tt.strftime("%Y-%m-%d") for tt in t]


def add_date_axis_on_top(ax, mjd_values: np.ndarray, n_ticks: int = 8) -> None:
    """Add a secondary x-axis on top of *ax* showing calendar dates."""
    finite = mjd_values[np.isfinite(mjd_values)]
    if len(finite) < 2:
        return
    mjd_lo, mjd_hi = float(finite.min()), float(finite.max())
    if mjd_hi <= mjd_lo:
        return
    n_ticks = max(3, min(n_ticks, len(finite)))
    tick_mjd = np.linspace(mjd_lo, mjd_hi, n_ticks)
    tick_lbls = mjd_to_datestr(tick_mjd)
    ax_top = ax.twiny()
    ax_top.set_xlim(ax.get_xlim())
    ax_top.set_xticks(tick_mjd)
    ax_top.set_xticklabels(tick_lbls, rotation=40, ha="left", fontsize=7)
    ax_top.tick_params(axis="x", length=4, pad=2)
    ax_top.set_xlabel("Date (UTC)", fontsize=7, labelpad=6)


def parse_bool(val) -> bool:
    """Coerce a dipole flag (bool / int / str / NaN) to Python bool."""
    if isinstance(val, bool):
        return val
    if isinstance(val, (int, float)) and np.isfinite(float(val)):
        return bool(int(val))
    if isinstance(val, str):
        return val.strip().lower() in ("true", "1", "yes")
    return False


print("Utility functions defined.")

## 3. Load data from notebook-01 outputs

In [ ]:
# ── diaObject catalogue (one row per object) ──────────────────────────────────
df_obj = pd.read_csv(FILE_DIAOBJ)
print(f"diaobj_catalogue : {len(df_obj):,} objects   fields: {sorted(df_obj['field'].unique())}")
print(f"Groups available : {sorted(df_obj['group'].unique())}")
df_obj.head(3)

In [ ]:
# ── Visit index (one row per object × visit) ──────────────────────────────────
df_visit = pd.read_csv(FILE_VISIT)
print(f"visit_index : {len(df_visit):,} rows")
df_visit.head(3)

In [ ]:
# ── Per-category diaSources parquets (visit-level, with dipole columns) ────────
# We load only the three Gaia categories that are of interest for calibration.
# These parquets contain one row per (diaObjectId, visit) with all dipole metrics.

src_frames = []
for group, path in SRC_FILES.items():
    if os.path.exists(path):
        dfi = pd.read_parquet(path)
        dfi["gaia_group"] = group  # tag the source group
        src_frames.append(dfi)
        print(f"  [{group:48s}] : {len(dfi):6,} diaSources   cols={list(dfi.columns[:8])}")
    else:
        print(f"  [WARNING] {path} not found — skipping.")

if src_frames:
    df_src = pd.concat(src_frames, ignore_index=True)
    print(f"\nTotal diaSources loaded : {len(df_src):,}")
else:
    df_src = pd.DataFrame()
    print("No diaSource parquets found.")

In [ ]:
# ── Add 'field' column to df_src by joining with df_obj ──────────────────────
# The parquets from notebook 01 may not have a field column directly;
# we join on diaObjectId.
if not df_src.empty and "diaObjectId" in df_src.columns and "field" not in df_src.columns:
    field_map = df_obj.set_index("diaObjectId")["field"]
    df_src["field"] = df_src["diaObjectId"].map(field_map)
    print(f"field column added to df_src from diaobj_catalogue.")

print(f"Fields in df_src : {sorted(df_src['field'].dropna().unique()) if not df_src.empty else 'N/A'}")

In [ ]:
# ── Quick summary per DDF ─────────────────────────────────────────────────────
if not df_src.empty:
    # Normalise isDipole column
    if "r:isDipole" in df_src.columns:
        df_src["isDipole"] = df_src["r:isDipole"].apply(parse_bool)
    elif "isDipole" not in df_src.columns:
        df_src["isDipole"] = False

    # Normalise dipoleAngle
    angle_col_candidates = ["r:dipoleAngle", "dipoleAngle"]
    ANGLE_COL = next((c for c in angle_col_candidates if c in df_src.columns), None)
    if ANGLE_COL:
        df_src[ANGLE_COL] = pd.to_numeric(df_src[ANGLE_COL], errors="coerce")
        print(f"Dipole angle column : {ANGLE_COL}")
    else:
        print("WARNING: no dipoleAngle column found in diaSources — rose plots will use diaobj catalogue.")

    # Normalise dipoleLength
    length_col_candidates = ["r:dipoleLength", "dipoleLength"]
    LENGTH_COL = next((c for c in length_col_candidates if c in df_src.columns), None)

    # MJD column
    mjd_col_candidates = ["r:midpointMjdTai", "midpointMjdTai"]
    MJD_COL = next((c for c in mjd_col_candidates if c in df_src.columns), None)
    print(f"MJD column  : {MJD_COL}")
    print(f"Length col  : {LENGTH_COL}")

    # RA/Dec columns
    ra_col = next((c for c in ["r:ra", "ra"] if c in df_src.columns), None)
    dec_col = next((c for c in ["r:dec", "dec"] if c in df_src.columns), None)
    print(f"RA  column  : {ra_col}")
    print(f"Dec column  : {dec_col}")

    # band column
    band_col_candidates = ["r:band", "band"]
    BAND_COL = next((c for c in band_col_candidates if c in df_src.columns), None)
    print(f"Band column : {BAND_COL}")

    print("\nSummary per DDF:")
    grp = df_src.groupby("field")
    for fname, sub in grp:
        n_dip = int(sub["isDipole"].sum())
        print(
            f"  {fname:12s}: {len(sub):6,} diaSources   {n_dip:5,} dipoles ({100 * n_dip / max(1, len(sub)):.1f}%)"
        )
else:
    ANGLE_COL = None
    LENGTH_COL = None
    MJD_COL = None
    ra_col = None
    dec_col = None
    BAND_COL = None

## 4(a) — Alert count maps and dipole sky maps per DDF

For each DDF one figure with two panels:
- **Left** : spatial distribution of all DIA sources colour-coded by Gaia group.
- **Right** : same field showing only dipole-flagged alerts as colour-coded arrows  
  (band colour, direction = `dipoleAngle`, length ∝ `dipoleLength`).

In [ ]:
def plot_skymap_and_dipoles(df_field: pd.DataFrame, field_name: str, max_arrows: int = MAX_ARROWS) -> None:
    """
    Two-panel figure for one DDF:
    Left  — all DIA sources coloured by Gaia group.
    Right — dipole alerts as arrows coloured by band.

    Parameters
    ----------
    df_field   : DataFrame of diaSources for this DDF (already filtered)
    field_name : label for title and file name
    max_arrows : max number of arrows drawn (performance cap)
    """
    if df_field.empty:
        print(f"[{field_name}] No data — skipping sky map.")
        return

    if ra_col is None or dec_col is None:
        print(f"[{field_name}] No RA/Dec columns — skipping sky map.")
        return

    df = df_field.copy()
    df["isDipole"] = df["isDipole"].fillna(False).astype(bool)
    n_total = len(df)
    n_dipoles = int(df["isDipole"].sum())

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # ── Panel LEFT: all alerts colour-coded by Gaia group ────────────────────
    ax_l = axes[0]
    for ggroup, sub in df.groupby("gaia_group"):
        ax_l.scatter(
            sub[ra_col].values,
            sub[dec_col].values,
            s=2,
            alpha=0.5,
            color=GROUP_COLORS.get(ggroup, "grey"),
            label=f"{GROUP_LABELS.get(ggroup, ggroup)} ({len(sub):,})",
            rasterized=True,
        )
    ax_l.invert_xaxis()  # standard RA orientation
    ax_l.set_xlabel("RA (deg)")
    ax_l.set_ylabel("Dec (deg)")
    ax_l.set_title(f"{field_name} — all DIA sources (n={n_total:,})", fontsize=10)
    ax_l.legend(loc="best", fontsize=7, markerscale=3, framealpha=0.7)

    # ── Panel RIGHT: dipole arrows coloured by band ───────────────────────────
    ax_r = axes[1]

    # Background: non-dipole points (light grey)
    df_nd = df[~df["isDipole"]]
    ax_r.scatter(
        df_nd[ra_col].values,
        df_nd[dec_col].values,
        s=1,
        c="#cccccc",
        alpha=0.3,
        label=f"non-dipole ({len(df_nd):,})",
        rasterized=True,
    )

    # Dipole arrows
    df_dp = df[df["isDipole"]]
    if len(df_dp) > 0 and ANGLE_COL and LENGTH_COL:
        df_plot = df_dp.dropna(subset=[ra_col, dec_col, ANGLE_COL, LENGTH_COL])
        if len(df_plot) > max_arrows:
            df_plot = df_plot.sample(max_arrows, random_state=42)
            print(f"  [{field_name}] Downsampled dipole arrows to {max_arrows}.")

        angle_rad = np.radians(df_plot[ANGLE_COL].values)
        length_deg = df_plot[LENGTH_COL].values / 3600.0  # arcsec → deg
        median_len = np.nanmedian(length_deg)
        scale = 0.05 / median_len if median_len > 0 else 1.0
        # PA convention: North=0, East=90, CCW on sky
        # In plot space: dx = -sin(PA) (East left), dy = cos(PA) (North up)
        dx = np.sin(angle_rad) * length_deg * scale
        dy = np.cos(angle_rad) * length_deg * scale

        if BAND_COL and BAND_COL in df_plot.columns:
            for band, grp in df_plot.groupby(BAND_COL):
                idx = df_plot.index.get_indexer(grp.index)
                ax_r.quiver(
                    grp[ra_col].values,
                    grp[dec_col].values,
                    dx[idx],
                    dy[idx],
                    color=BAND_COLORS.get(band, "red"),
                    angles="xy",
                    scale_units="xy",
                    scale=1.0,
                    width=0.002,
                    headwidth=4,
                    headlength=4,
                    label=f"dipole band={band} ({len(grp):,})",
                    alpha=0.75,
                )
        else:
            ax_r.quiver(
                df_plot[ra_col].values,
                df_plot[dec_col].values,
                dx,
                dy,
                color="red",
                angles="xy",
                scale_units="xy",
                scale=1.0,
                width=0.002,
                headwidth=4,
                headlength=4,
                label=f"dipole ({len(df_plot):,})",
                alpha=0.75,
            )
    elif len(df_dp) > 0:
        ax_r.scatter(
            df_dp[ra_col].values,
            df_dp[dec_col].values,
            s=8,
            c="red",
            alpha=0.5,
            label=f"dipole ({len(df_dp):,})",
        )

    ax_r.invert_xaxis()
    ax_r.set_xlabel("RA (deg)")
    ax_r.set_ylabel("Dec (deg)")
    ax_r.set_title(
        f"{field_name} — dipole alerts as arrows\n"
        f"total={n_total:,}  dipoles={n_dipoles:,}  ({100 * n_dipoles / max(1, n_total):.1f}%)",
        fontsize=10,
    )
    ax_r.legend(loc="best", fontsize=7, markerscale=2, framealpha=0.7)

    fig.suptitle(
        f"DIA source distribution and dipole map — {field_name}",
        fontsize=12,
        y=1.01,
    )
    plt.tight_layout()
    savefig(f"skymap_and_dipoles_{field_name.replace('-', '_')}")
    plt.show()
    # plt.close(fig)


print("plot_skymap_and_dipoles() defined.")

In [ ]:
# ── Run sky maps for all DDFs ────────────────────────────────────────────────
if not df_src.empty:
    fields_present = sorted(df_src["field"].dropna().unique())
    print(f"DDFs with data: {fields_present}")
    for fname in fields_present:
        df_f = df_src[df_src["field"] == fname].copy()
        plot_skymap_and_dipoles(df_f, fname)
else:
    print("No diaSource data — skipping sky maps.")

## 4(b) — Dipole fraction vs MJD (and calendar date)

Per-night dipole fraction (1-day bins) for all DDFs.  
Grey bars (log scale, right axis) show the number of alerts per bin.  
Calendar-date axis is drawn on top of the first panel.

In [ ]:
def compute_dipole_fraction_vs_time(
    df: pd.DataFrame,
    bin_days: float = MJD_BIN_DAYS,
) -> pd.DataFrame:
    """
    Bin diaSources by MJD and compute the dipole fraction per bin.

    Returns a DataFrame with columns:
        mjd_bin_center, n_total, n_dipoles, dipole_fraction, dipole_fraction_err
    """
    if df.empty or MJD_COL is None or MJD_COL not in df.columns:
        return pd.DataFrame()

    df = df.copy()
    df["isDipole"] = df["isDipole"].fillna(False).astype(bool)
    df[MJD_COL] = pd.to_numeric(df[MJD_COL], errors="coerce")
    df = df.dropna(subset=[MJD_COL])
    if df.empty:
        return pd.DataFrame()

    mjd_min = df[MJD_COL].min()
    mjd_max = df[MJD_COL].max()
    bins = np.arange(mjd_min, mjd_max + bin_days, bin_days)
    df["mjd_bin"] = pd.cut(df[MJD_COL], bins=bins, labels=False)
    bin_centers = (bins[:-1] + bins[1:]) / 2.0

    rows = []
    for i, center in enumerate(bin_centers):
        mask = df["mjd_bin"] == i
        sub = df[mask]
        n_tot = len(sub)
        n_dip = int(sub["isDipole"].sum())
        frac = n_dip / n_tot if n_tot > 0 else np.nan
        frac_e = np.sqrt(n_dip) / n_tot if (n_tot > 0 and n_dip > 0) else np.nan
        rows.append(
            {
                "mjd_bin_center": center,
                "n_total": n_tot,
                "n_dipoles": n_dip,
                "dipole_fraction": frac,
                "dipole_fraction_err": frac_e,
            }
        )
    return pd.DataFrame(rows)


print("compute_dipole_fraction_vs_time() defined.")

In [ ]:
# ── Stacked multi-panel figure: one subplot per DDF ──────────────────────────
if df_src.empty or MJD_COL is None:
    print("No diaSource data or MJD column not found — skipping fraction-vs-time plots.")
else:
    fields_present = sorted(df_src["field"].dropna().unique())
    n_ddf = len(fields_present)

    fig, axes = plt.subplots(
        nrows=n_ddf,
        ncols=1,
        figsize=(12, 2.8 * n_ddf),
        sharex=False,
    )
    if n_ddf == 1:
        axes = [axes]

    for ax_i, fname in enumerate(fields_present):
        df_f = df_src[df_src["field"] == fname].copy()
        df_time = compute_dipole_fraction_vs_time(df_f, bin_days=MJD_BIN_DAYS)
        ax = axes[ax_i]

        if df_time.empty:
            ax.set_title(f"{fname} — no data")
            continue

        mjd = df_time["mjd_bin_center"].values
        frac = df_time["dipole_fraction"].values * 100.0
        err = df_time["dipole_fraction_err"].fillna(0).values * 100.0

        ax.errorbar(
            mjd,
            frac,
            yerr=err,
            fmt="o-",
            ms=4,
            lw=1.2,
            capsize=3,
            color="steelblue",
            label=f"bin = {MJD_BIN_DAYS} d",
        )

        # Twin axis: N alerts per bin (log scale)
        ax2 = ax.twinx()
        ax2.bar(
            mjd,
            df_time["n_total"].values,
            width=MJD_BIN_DAYS * 0.8,
            color="grey",
            alpha=0.35,
            label="N alerts",
        )
        ax2.set_ylabel("N alerts", fontsize=7, color="grey")
        ax2.tick_params(axis="y", labelcolor="grey", labelsize=8)
        ax2.set_yscale("log")

        ax.set_ylabel("Dipole fraction (%)", fontsize=9)
        ax.text(
            0.02,
            0.93,
            f"DDF : {fname}",
            transform=ax.transAxes,
            fontsize=10,
            va="top",
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.5),
        )
        ax.legend(loc="upper right", fontsize=8)

        # Calendar date axis on top of the FIRST subplot only
        if ax_i == 0:
            add_date_axis_on_top(ax, mjd, n_ticks=10)

        if ax_i == n_ddf - 1:
            ax.set_xlabel("MJD (TAI)", fontsize=9)

    fig.suptitle(
        f"Dipole fraction vs time — all DDFs  (bin = {MJD_BIN_DAYS} day)",
        fontsize=12,
        y=1.005,
    )
    plt.tight_layout()
    savefig(f"dipole_fraction_vs_time_bin{MJD_BIN_DAYS}d_all_ddfs")
    plt.show()
    # plt.close(fig)

## 4(c) — Rose plots per DDF (all bands overlaid in one rose)

For each DDF one figure with two panels:
- **Left** — polar rose diagram with all bands stacked (stacked angular histogram).
  The dashed red circle marks the uniform-distribution level.
- **Right** — same data as a linear stacked histogram of `dipoleAngle` (0–360°),  
  each band colour-coded.

Only dipole-flagged diaSources (`isDipole=True`) are included.

In [ ]:
def plot_rose_and_linear_per_ddf(
    df_field: pd.DataFrame,
    field_name: str,
    n_bins: int = N_BINS_ROSE,
) -> None:
    """
    Rose plot (polar) + linear histogram of dipoleAngle for one DDF,
    with all filter bands overlaid inside the same plot.

    Parameters
    ----------
    df_field   : diaSources for this DDF, already filtered by isDipole=True
    field_name : label for title / file name
    n_bins     : number of angular bins (default 36 = 10°/bin)
    """
    if df_field.empty or ANGLE_COL is None:
        print(f"[{field_name}] No angle data — skipping rose plot.")
        return

    df_dip = df_field[df_field["isDipole"]].copy()
    angles_all_deg = pd.to_numeric(df_dip[ANGLE_COL], errors="coerce").dropna().values % 360.0

    if len(angles_all_deg) == 0:
        print(f"[{field_name}] No finite dipoleAngle values — skipping.")
        return

    bin_edges_deg = np.linspace(0, 360, n_bins + 1)
    bin_edges_rad = np.deg2rad(bin_edges_deg)
    bin_centers_rad = (bin_edges_rad[:-1] + bin_edges_rad[1:]) / 2.0
    bin_width_rad = 2 * np.pi / n_bins

    # Determine bands present
    if BAND_COL and BAND_COL in df_dip.columns:
        bands_present = [b for b in BAND_ORDER if b in df_dip[BAND_COL].dropna().unique()]
    else:
        bands_present = []

    fig = plt.figure(figsize=(12, 5))
    ax_pol = fig.add_subplot(1, 2, 1, projection="polar")
    ax_lin = fig.add_subplot(1, 2, 2)

    # ── Polar rose — stacked bands ─────────────────────────────────────────
    bottom_pol = np.zeros(n_bins)
    bottom_lin = np.zeros(n_bins)

    if bands_present:
        for band in bands_present:
            sub = (
                pd.to_numeric(df_dip.loc[df_dip[BAND_COL] == band, ANGLE_COL], errors="coerce")
                .dropna()
                .values
                % 360.0
            )
            if len(sub) == 0:
                continue
            bc = BAND_COLORS.get(band, "grey")
            cnts, _ = np.histogram(sub, bins=bin_edges_deg)
            # Polar
            ax_pol.bar(
                bin_centers_rad,
                cnts,
                width=bin_width_rad * 0.90,
                bottom=bottom_pol,
                color=bc,
                edgecolor="white",
                linewidth=0.3,
                alpha=0.85,
                label=f"{band} (n={len(sub):,})",
            )
            bottom_pol = bottom_pol + cnts
            # Linear
            ax_lin.bar(
                bin_edges_deg[:-1],
                cnts,
                width=360.0 / n_bins * 0.90,
                bottom=bottom_lin,
                color=bc,
                edgecolor="white",
                linewidth=0.3,
                alpha=0.85,
                label=f"{band} (n={len(sub):,})",
            )
            bottom_lin = bottom_lin + cnts
    else:
        # No band information: plot all together
        cnts, _ = np.histogram(angles_all_deg, bins=bin_edges_deg)
        ax_pol.bar(
            bin_centers_rad,
            cnts,
            width=bin_width_rad * 0.90,
            bottom=0,
            color="steelblue",
            edgecolor="white",
            linewidth=0.3,
            alpha=0.85,
            label=f"all bands (n={len(angles_all_deg):,})",
        )
        ax_lin.bar(
            bin_edges_deg[:-1],
            cnts,
            width=360.0 / n_bins * 0.90,
            bottom=0,
            color="steelblue",
            edgecolor="white",
            linewidth=0.3,
            alpha=0.85,
            label=f"all bands (n={len(angles_all_deg):,})",
        )

    # Dashed reference circle at the uniform level
    total_counts = bottom_pol  # cumulative heights after all bands
    max_count = float(np.max(total_counts)) if total_counts.max() > 0 else 1.0
    uniform_val = len(angles_all_deg) / n_bins
    ax_pol.plot(
        np.linspace(0, 2 * np.pi, 300),
        np.full(300, uniform_val),
        "--",
        color="crimson",
        lw=1.2,
        alpha=0.85,
        label="uniform",
    )

    # Polar axis decoration
    ax_pol.set_theta_zero_location("N")  # North at top
    ax_pol.set_theta_direction(-1)  # clockwise (PA convention)
    ax_pol.tick_params(labelsize=7)
    ax_pol.set_title(
        f"{field_name} — polar rose (n={len(angles_all_deg):,})",
        va="bottom",
        pad=18,
        fontsize=10,
    )
    # Cardinal direction labels
    r_lbl = ax_pol.get_rmax() * 1.22
    for pa_deg, lbl in [(0, "N"), (90, "E"), (180, "S"), (270, "W")]:
        ax_pol.text(
            np.radians(pa_deg),
            r_lbl,
            lbl,
            ha="center",
            va="center",
            fontsize=8,
            fontweight="bold",
        )
    ax_pol.legend(loc="lower right", fontsize=7, bbox_to_anchor=(1.30, -0.05))

    # Linear axis decoration
    ax_lin.set_xlabel("Dipole angle (deg, N through E)", fontsize=9)
    ax_lin.set_ylabel("N dipole alerts", fontsize=9)
    ax_lin.set_xlim(0, 360)
    ax_lin.set_xticks(np.arange(0, 361, 45))
    ax_lin.set_xticklabels(
        ["N\n0°", "45°", "E\n90°", "135°", "S\n180°", "225°", "W\n270°", "315°", "N\n360°"],
        fontsize=8,
    )
    ax_lin.set_title(f"{field_name} — dipoleAngle by band (linear)", fontsize=10)
    ax_lin.legend(loc="upper right", fontsize=8, ncol=2)

    fig.suptitle(
        f"Dipole orientation — {field_name}  (isDipole=True only)",
        fontsize=12,
        y=1.02,
    )
    plt.tight_layout()
    savefig(f"rose_dipoleAngle_{field_name.replace('-', '_')}")
    plt.show()
    # plt.close(fig)


print("plot_rose_and_linear_per_ddf() defined.")

In [ ]:
# ── Run rose plots for all DDFs ──────────────────────────────────────────────
if df_src.empty or ANGLE_COL is None:
    print("No diaSource data or angle column not found — skipping rose plots.")
else:
    fields_present = sorted(df_src["field"].dropna().unique())
    for fname in fields_present:
        df_f = df_src[df_src["field"] == fname].copy()
        plot_rose_and_linear_per_ddf(df_f, fname)

## 5. Summary: dipole fraction per DDF × band (heat map)

2D matrix `field × band` of dipole fraction,
as a visual complement to the time-resolved figures above.

In [ ]:
if not df_src.empty and BAND_COL:
    band_rows = []
    for fname in sorted(df_src["field"].dropna().unique()):
        df_f = df_src[df_src["field"] == fname].copy()
        df_f["isDipole"] = df_f["isDipole"].fillna(False).astype(bool)
        for band, grp in df_f.groupby(BAND_COL):
            n_tot = len(grp)
            n_dip = int(grp["isDipole"].sum())
            band_rows.append(
                {
                    "field": fname,
                    "band": band,
                    "n_total": n_tot,
                    "n_dipoles": n_dip,
                    "dipole_fraction": n_dip / n_tot if n_tot > 0 else np.nan,
                }
            )

    df_band = pd.DataFrame(band_rows)
    pivot = df_band.pivot_table(index="field", columns="band", values="dipole_fraction").reindex(
        columns=[b for b in BAND_ORDER if b in df_band["band"].unique()]
    )
    print("Dipole fraction per field × band (%):\n")
    print((pivot * 100).to_string(float_format="{:.2f}".format))

    fig, ax = plt.subplots(figsize=(8, max(3, len(pivot) * 0.7)))
    im = ax.imshow(pivot.values * 100, aspect="auto", cmap="YlOrRd", vmin=0)
    ax.set_xticks(range(pivot.shape[1]))
    ax.set_xticklabels(pivot.columns.tolist(), fontsize=10)
    ax.set_yticks(range(pivot.shape[0]))
    ax.set_yticklabels(pivot.index.tolist(), fontsize=9)
    ax.set_xlabel("Band", fontsize=10)
    ax.set_ylabel("DDF", fontsize=10)
    ax.set_title("Dipole fraction (%) per DDF × band", fontsize=11)
    plt.colorbar(im, ax=ax, label="Dipole fraction (%)")
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            val = pivot.values[i, j]
            if not np.isnan(val):
                ax.text(j, i, f"{val * 100:.1f}", ha="center", va="center", fontsize=8)
    plt.tight_layout()
    savefig("dipole_fraction_heatmap_field_x_band")
    plt.show()
    # plt.close(fig)
else:
    print("No data available for heatmap.")

## 6. Dipole fraction per DDF — global bar chart

Bar chart of the global (all-bands) dipole fraction per DDF,
coloured by field.

In [ ]:
if not df_src.empty:
    summary_rows = []
    for fname in sorted(df_src["field"].dropna().unique()):
        df_f = df_src[df_src["field"] == fname].copy()
        df_f["isDipole"] = df_f["isDipole"].fillna(False).astype(bool)
        n_tot = len(df_f)
        n_dip = int(df_f["isDipole"].sum())
        summary_rows.append(
            {
                "field": fname,
                "n_total": n_tot,
                "n_dipoles": n_dip,
                "dipole_fraction": n_dip / n_tot if n_tot > 0 else np.nan,
            }
        )
    df_summary = pd.DataFrame(summary_rows)
    print(df_summary.to_string(index=False))

    fig, ax = plt.subplots(figsize=(9, 4))
    cmap = cm.tab10
    colors = [cmap(i / max(len(df_summary) - 1, 1)) for i in range(len(df_summary))]
    bars = ax.bar(
        df_summary["field"],
        df_summary["dipole_fraction"].fillna(0) * 100,
        color=colors,
        edgecolor="k",
        linewidth=0.5,
    )
    ax.set_ylabel("Dipole fraction (%)")
    ax.set_title("Fraction of DIA alerts flagged as dipoles per DDF (Gaia categories only)")
    ax.tick_params(axis="x", rotation=30)
    for bar, frac in zip(bars, df_summary["dipole_fraction"].fillna(0) * 100):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.05,
            f"{frac:.1f}%",
            ha="center",
            va="bottom",
            fontsize=8,
        )
    plt.tight_layout()
    savefig("dipole_fraction_per_ddf_barchart")
    plt.show()
    # plt.close(fig)
else:
    print("No data available for bar chart.")

## 7. All-DDF rose plots in one figure (global overview)

One polar subplot per DDF arranged in a grid, bands stacked inside each rose.

In [ ]:
if not df_src.empty and ANGLE_COL:
    fields_present = sorted(df_src["field"].dropna().unique())
    n_fields = len(fields_present)
    n_cols = min(4, n_fields)
    n_rows = int(np.ceil(n_fields / n_cols))

    bin_edges_deg = np.linspace(0, 360, N_BINS_ROSE + 1)
    bin_edges_rad = np.deg2rad(bin_edges_deg)
    bin_centers_rad = (bin_edges_rad[:-1] + bin_edges_rad[1:]) / 2.0
    bin_width_rad = 2 * np.pi / N_BINS_ROSE

    fig = plt.figure(figsize=(4.5 * n_cols, 4.5 * n_rows))

    for idx, fname in enumerate(fields_present):
        ax_pol = fig.add_subplot(n_rows, n_cols, idx + 1, projection="polar")
        df_f = df_src[(df_src["field"] == fname) & df_src["isDipole"]].copy()

        if df_f.empty or ANGLE_COL not in df_f.columns:
            ax_pol.set_title(f"{fname}\n(no data)", va="bottom", pad=18, fontsize=9)
            continue

        bottom_pol = np.zeros(N_BINS_ROSE)
        n_total_dip = 0

        if BAND_COL and BAND_COL in df_f.columns:
            bands_present = [b for b in BAND_ORDER if b in df_f[BAND_COL].dropna().unique()]
            for band in bands_present:
                sub = (
                    pd.to_numeric(df_f.loc[df_f[BAND_COL] == band, ANGLE_COL], errors="coerce")
                    .dropna()
                    .values
                    % 360.0
                )
                if len(sub) == 0:
                    continue
                n_total_dip += len(sub)
                cnts, _ = np.histogram(sub, bins=bin_edges_deg)
                ax_pol.bar(
                    bin_centers_rad,
                    cnts,
                    width=bin_width_rad * 0.88,
                    bottom=bottom_pol,
                    color=BAND_COLORS.get(band, "grey"),
                    edgecolor="white",
                    linewidth=0.2,
                    alpha=0.85,
                    label=band,
                )
                bottom_pol += cnts
        else:
            angles_all = pd.to_numeric(df_f[ANGLE_COL], errors="coerce").dropna().values % 360.0
            n_total_dip = len(angles_all)
            cnts, _ = np.histogram(angles_all, bins=bin_edges_deg)
            ax_pol.bar(
                bin_centers_rad,
                cnts,
                width=bin_width_rad * 0.88,
                bottom=0,
                color="steelblue",
                edgecolor="white",
                linewidth=0.2,
                alpha=0.85,
            )

        # Uniform reference
        uniform_val = n_total_dip / N_BINS_ROSE
        ax_pol.plot(
            np.linspace(0, 2 * np.pi, 300),
            np.full(300, uniform_val),
            "--",
            color="crimson",
            lw=0.9,
            alpha=0.8,
        )

        ax_pol.set_theta_zero_location("N")
        ax_pol.set_theta_direction(-1)
        ax_pol.tick_params(labelsize=6)
        ax_pol.set_title(f"{fname}\n(n_dip={n_total_dip:,})", va="bottom", pad=14, fontsize=9)

    # Shared legend (band colours) in the last cell or as fig legend
    handles = [
        plt.Rectangle((0, 0), 1, 1, color=BAND_COLORS.get(b, "grey"), alpha=0.85, label=b) for b in BAND_ORDER
    ]
    handles.append(plt.Line2D([0], [0], ls="--", color="crimson", lw=1.2, label="uniform"))
    fig.legend(
        handles=handles,
        loc="lower center",
        ncol=len(BAND_ORDER) + 1,
        fontsize=9,
        bbox_to_anchor=(0.5, -0.02),
        frameon=True,
    )

    fig.suptitle(
        "Dipole orientation rose plots — all DDFs  (bands stacked, isDipole=True)",
        fontsize=13,
        y=1.01,
    )
    plt.tight_layout()
    savefig("rose_dipoleAngle_all_ddfs_overview")
    plt.show()
    # plt.close(fig)
else:
    print("No data for global rose-plot overview.")

## 8. Interpretation notes

### Rose plots

- **Uniform distribution** (dashed red circle) → dipole orientations are **random** → dipoles are consistent with noise or random mis-registration.
- **Preferred angle** → systematic effect at the telescope or detector level (tracking error, read-out axis, wind shake, DCR, guider …).
- **Band dependence**: if different bands show different preferred directions, that points to a wavelength-dependent effect (DCR, PSF chromaticity, …).

### Fraction vs time

- A **decreasing trend** over time would indicate that the Rubin AP pipeline improved its template co-registration during commissioning.
- **Sudden changes** (steps) may correlate with pipeline version updates or changes in the coadd template.
- **Night-to-night scatter** may reflect seeing variability, airmass, or sky-background changes.

### dipoleAngle convention

The Rubin AP pipeline stores `dipoleAngle` as the **position angle of the positive lobe**,
measured **East of North** in degrees (standard astronomical PA).  
A dipole is physically undirected (period 180°); if you want the undirected orientation,
use `dipoleAngle % 180`.


In [ ]:
print("Notebook 11g complete.")
print(f"Figures saved to: {os.path.abspath(DIR_FIGS)}")